# 02_synthval: Check the synthetic data
Reads the 4 files from Notebook 1 (`output/`). Run Notebook 1 first. After **Restart**, use **Run All**.

Order in every part: CODE → TEST → RUN.

## Setup

In [1]:
import numpy as np, pandas as pd
import ipytest
# raise_on_error: a test cell that fails or collects no tests stops the run.
# addopts without the default "-q", so each cell shows "N passed"; no colour codes in saved outputs.
ipytest.autoconfig(raise_on_error=True, addopts=["--color=no", "-p", "no:cacheprovider",
                                                 "-W", "ignore::pytest.PytestAssertRewriteWarning"])
print("Setup OK")

Setup OK


## Cell 1: Load the files from Notebook 1

In [2]:
# CODE
# ---------------------------------------------------------------------------
# Cell 1: load the 4 files from Notebook 1
# ---------------------------------------------------------------------------
def load_outputs(folder="output"):
    patients = pd.read_csv(f"{folder}/patients.csv")
    relatives = pd.read_csv(f"{folder}/relatives.csv")
    visits = pd.read_csv(f"{folder}/visits.csv", parse_dates=["visit_date"])
    training = pd.read_csv(f"{folder}/training.csv", parse_dates=["index_date"])
    return patients, relatives, visits, training

In [3]:
# RUN
import os
FOLDER = os.environ.get("SYNTH_DIR", "output")          # "output_seed2" for the second seed
print("data folder:", FOLDER)
PATIENTS, RELATIVES, VISITS, TRAINING = load_outputs(FOLDER)
for name, df in [("patients", PATIENTS), ("relatives", RELATIVES), ("visits", VISITS), ("training", TRAINING)]:
    print(f"{name}: {len(df):,} rows")

data folder: output


patients: 20,000 rows
relatives: 242,876 rows
visits: 131,953 rows
training: 70,297 rows


## Cell 2: Hard rules (must be 0 failures)
The 10 rules from the System Design, Section 7. Ranges come from the CDSS Data Dictionary and are typed here again on purpose, so a mistake in Notebook 1 cannot hide itself.

In [4]:
# CODE
# ---------------------------------------------------------------------------
# Cell 2: the 10 hard rules (System Design, Section 7)
# Min/max copied from the CDSS Data Dictionary (oct31), Parameter Ingestion Tables.
# Written here again ON PURPOSE (not imported from Notebook 1), so a mistake in
# the generator's table cannot hide itself.
# ---------------------------------------------------------------------------
RANGES = {
    "age": (18, 100),
    # CVD
    "sbp": (70, 220), "dbp": (40, 130), "heart_rate": (40, 180), "bmi": (12.0, 60.0),
    "total_cholesterol": (100, 400), "hdl": (15, 120), "ldl": (30, 300), "triglycerides": (40, 1000),
    "hs_crp": (0.1, 50.0), "creatinine": (0.4, 10.0), "egfr": (5, 140), "lvef": (15, 75),
    "ctni": (0.01, 50.0), "nt_probnp": (10, 35000), "pack_years": (0, 120),
    "sodium_mg": (500, 10000), "activity_min_week": (0, 1050),
    # T2D
    "fpg": (50, 450), "pg_2h": (60, 600), "hba1c": (4.0, 15.0), "insulin": (1.0, 100.0),
    "c_peptide": (0.1, 12.0), "homa_ir": (0.2, 25.0), "waist_cm": (50, 160), "whr": (0.60, 1.40),
    "uacr": (1, 3500), "uric_acid": (1.5, 14.0), "carbs_g": (50, 600), "sedentary_hours": (1.0, 18.0),
    "gestational_diabetes": (0, 1), "acanthosis_nigricans": (0, 1),
    # CRD
    "fev1": (0.5, 5.5), "fvc": (1.0, 7.0), "fev1_fvc": (25.0, 95.0), "fev1_pct_pred": (15.0, 120.0),
    "pef": (50, 800), "eosinophils": (0, 2500), "feno": (5, 150), "spo2": (65.0, 100.0),
    "pao2": (40, 110), "paco2": (25, 90), "mmrc": (0, 4), "act": (5, 25), "cat": (0, 40),
    "ige": (2, 3000), "exacerbations_year": (0, 15), "occupational_dust": (0, 1),
    "allergic_rhinitis": (0, 1), "childhood_asthma": (0, 1),
    # NEURO
    "phq9": (0, 27), "gad7": (0, 21), "psqi": (0, 21), "sleep_hours": (1.0, 16.0),
    "alcohol_use": (0, 1), "tbi_history": (0, 1), "tsh": (0.01, 100.0), "vitamin_b12": (50, 2000),
    # FHRS: 0, no fixed maximum
    "fhrs_CVD": (0.0, np.inf), "fhrs_T2D": (0.0, np.inf), "fhrs_CRD": (0.0, np.inf), "fhrs_NEURO": (0.0, np.inf),
}

# FHRS, copied from Notebook 1 Part A (same FHRS Guide numbers)
FHRS_CFG = {"R": {1: 0.50, 2: 0.25, 3: 0.125}, "S": {"confirmed": 1.0, "not_sure": 0.5}, "cluster_bonus": 1.0,
            "penalties": {"CVD": {"secondhand_smoke": 0.15, "high_salt_diet": 0.15},
                          "T2D": {"processed_diet": 0.15, "low_home_activity": 0.15},
                          "CRD": {"secondhand_smoke": 0.15, "indoor_mold": 0.15, "coal_heating": 0.15},
                          "NEURO": {}}}
DOMAIN = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def onset_multiplier(disease, age, sex):
    if age is None: return 1.0
    if disease == "CVD":
        if age < 40: return 3.0
        if sex == "M": return 2.0 if age < 55 else 1.0 if age < 65 else 0.75
        return 2.0 if age < 65 else 1.0 if age < 75 else 0.75
    if disease in ("T2D", "COPD"):
        return 3.0 if age < 40 else 2.0 if age < 50 else 1.0 if age < 65 else 0.75
    if disease == "asthma":
        return 3.0 if age < 12 else 2.0 if age <= 25 else 1.0 if age <= 40 else 0.75
    if disease == "depression":
        return 3.0 if age < 18 else 2.0 if age <= 24 else 1.0 if age <= 40 else 0.75
    return 1.0

def fhrs(relatives, exposures, domain, on_year, cfg=FHRS_CFG):
    bio, confirmed_first = 0.0, {}
    for r in relatives:
        for c in r["conditions"]:
            if c["disease"] in DOMAIN[domain] and c["year"] <= on_year:
                s = cfg["S"]["confirmed"] if c["confirmed"] else cfg["S"]["not_sure"]
                bio += cfg["R"][r["degree"]] * onset_multiplier(c["disease"], c["age"], r["sex"]) * s
                if r["degree"] == 1 and c["confirmed"]:
                    confirmed_first[c["disease"]] = confirmed_first.get(c["disease"], 0) + 1
                break
    if any(n >= 2 for n in confirmed_first.values()):
        bio += cfg["cluster_bonus"]
    E = 1.0 + sum(v for k, v in cfg["penalties"][domain].items() if exposures.get(k))
    return round(bio * E, 2)

def egfr_ckd_epi(creatinine, age, sex):
    """CKD-EPI 2021 race-free equation; sex 'Male'/'Female'."""
    female = sex in ("Female", "F")
    k, a = (0.7, -0.241) if female else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if female else 1)

def _family(relatives):
    """relatives.csv rows -> {patient_id: [relative dicts]} in the FHRS format."""
    fams = {}
    for (pid, _), g in relatives.groupby(["patient_id", "relative_no"], sort=False):
        first = g.iloc[0]
        conds = [{"disease": c.disease, "year": int(c.dx_year),
                  "age": None if pd.isna(c.dx_age) else int(c.dx_age),
                  "confirmed": str(c.confirmed) == "True"}
                 for c in g.itertuples()
                 if pd.notna(c.disease) and str(getattr(c, "diagnosed", True)) != "False"]   # patient can't know undiagnosed
        fams.setdefault(pid, []).append({"degree": int(first.degree), "sex": first.sex, "conditions": conds})
    return fams

def check_hard_rules(patients, relatives, visits, ranges=RANGES):
    """Returns {rule: number of failing rows}. Pass = every number is 0."""
    out = {}
    v = visits.merge(patients[["patient_id", "sex", "birth_year"]], on="patient_id", how="left")
    v = v.sort_values(["patient_id", "visit_no"])
    by_patient = v.groupby("patient_id")

    # 1. every value inside its dictionary min-max range (a missing column is a failure, not skipped)
    bad = pd.Series(False, index=v.index)
    for col, (lo, hi) in ranges.items():
        if col not in v.columns:
            out[f"1 column missing: {col}"] = 1
            continue
        bad |= v[col].notna() & ~v[col].between(lo, hi)
    out["1 value inside dictionary range"] = int(bad.sum())

    # 2. age matches birth year and visit date
    out["2 age matches birth year and visit date"] = int((v["age"] != v["visit_date"].dt.year - v["birth_year"]).sum())

    # 3. visit dates move forward
    out["3 visit dates move forward"] = int((by_patient["visit_date"].diff().dt.days <= 0).sum())

    # 4. pack-years never go down
    out["4 pack-years never go down"] = int((by_patient["pack_years"].diff() < 0).sum())

    # 5. a diagnosis, once recorded, stays
    out["5 diagnosis stays in later visits"] = int(sum((by_patient[f"has_{d}"].diff() < 0).sum() for d in DOMAIN))

    # 6. a relative's diagnosis falls inside their lifetime
    r = relatives.dropna(subset=["disease"])
    outside = (r["dx_year"] < r["birth_year"]) | (r["death_year"].notna() & (r["dx_year"] > r["death_year"]))
    out["6 relative diagnosis inside their lifetime"] = int(outside.sum())

    # 7. smoking status never goes back to "never"
    was_smoker = by_patient["smoking_status"].transform(lambda s: (s != "never").cummax().shift(fill_value=False))
    out["7 smoking never goes back to never"] = int((was_smoker & (v["smoking_status"] == "never")).sum())

    # 8. gestational diabetes only for females
    out["8 gestational diabetes only for females"] = int(((patients["sex"] == "Male") & (patients["gestational_diabetes"] == 1)).sum())

    # 9. parents older than patient; grandparents older than that parent; degree matches relation
    rel = relatives.drop_duplicates(["patient_id", "relative_no"]).merge(
        patients[["patient_id", "birth_year"]].rename(columns={"birth_year": "patient_by"}), on="patient_id")
    parents = rel[rel["relation"].isin(["father", "mother"])]
    out["9 parents older than patient"] = int((parents["birth_year"] >= parents["patient_by"]).sum())
    grand = rel[rel["relation"].str.contains("'s")].assign(parent=lambda d: d["relation"].str.split("'s").str[0])
    grand = grand.merge(parents[["patient_id", "relation", "birth_year"]].rename(
        columns={"relation": "parent", "birth_year": "parent_by"}), on=["patient_id", "parent"])
    out["9 grandparents older than parent"] = int((grand["birth_year"] >= grand["parent_by"]).sum())
    expected_degree = np.where(rel["relation"].str.contains("'s"), 2, 1)
    out["9 degree matches relation"] = int((rel["degree"] != expected_degree).sum())

    # 10. computed fields match their inputs
    def mismatch(col, expected, tol):
        has = v[col].notna() & expected.notna()
        return int((has & ((v[col] - expected).abs() > tol)).sum())
    out["10 computed field matches: bmi"] = mismatch("bmi", (v["weight_kg"] / (v["height_cm"] / 100) ** 2).round(1), 0.05)
    out["10 computed field matches: homa_ir"] = mismatch("homa_ir", (v["fpg"] * v["insulin"] / 405).clip(0.2, 25).round(1), 0.05)
    out["10 computed field matches: fev1_fvc"] = mismatch("fev1_fvc", (v["fev1"] / v["fvc"] * 100).round(1), 0.05)
    egfr = pd.Series([np.nan if pd.isna(c) else float(np.clip(round(egfr_ckd_epi(c, a, s)), 5, 140))
                      for c, a, s in zip(v["creatinine"], v["age"], v["sex"])], index=v.index)
    out["10 computed field matches: egfr"] = mismatch("egfr", egfr, 0.5)
    fams = _family(relatives)
    exp = patients.set_index("patient_id")[[c for c in patients if c.startswith("exp_")]]
    exposures = {pid: {k[4:]: bool(x) for k, x in row.items()} for pid, row in exp.iterrows()}
    for d in DOMAIN:
        expected = pd.Series([fhrs(fams.get(pid, []), exposures.get(pid, {}), d, dt.year)
                              for pid, dt in zip(v["patient_id"], v["visit_date"])], index=v.index)
        out[f"10 computed field matches: fhrs_{d}"] = mismatch(f"fhrs_{d}", expected, 0.005)
    return out

def print_report(results):
    failed = {k: n for k, n in results.items() if n > 0}
    for rule, n in results.items():
        print(f"{'PASS' if n == 0 else 'FAIL'}  {rule}" + (f"  ({n:,} rows)" if n else ""))
    print("\nALL HARD RULES PASS" if not failed else f"\n{len(failed)} RULE(S) FAILED")
    return not failed

In [5]:
%%ipytest -q
# TEST: a small clean record passes; breaking one rule makes exactly that rule fail
import pytest

SMALL_RANGES = {"sbp": (70, 220), "bmi": (12.0, 60.0), "age": (18, 100)}

def clean():
    patients = pd.DataFrame({"patient_id": [0], "sex": ["Female"], "birth_year": [1980],
                             "gestational_diabetes": [0], "exp_secondhand_smoke": [0],
                             "exp_high_salt_diet": [0], "exp_processed_diet": [0],
                             "exp_low_home_activity": [0], "exp_indoor_mold": [0], "exp_coal_heating": [0]})
    relatives = pd.DataFrame({
        "patient_id": [0, 0, 0], "relative_no": [1, 2, 3],
        "relation": ["father", "mother", "father's father"], "degree": [1, 1, 2],
        "sex": ["M", "F", "M"], "birth_year": [1950, 1952, 1920], "death_year": [None, None, 1990],
        "disease": ["CVD", None, None], "dx_year": [2010, None, None], "dx_age": [60, None, None],
        "confirmed": [True, None, None]})
    visits = pd.DataFrame({
        "patient_id": [0, 0], "visit_no": [1, 2],
        "visit_date": pd.to_datetime(["2021-03-01", "2022-03-01"]), "age": [41, 42],
        "sbp": [120, 125], "weight_kg": [60.0, 61.0], "height_cm": [160.0, 160.0],
        "bmi": [23.4, 23.8], "fpg": [90, float("nan")], "insulin": [10.0, float("nan")],
        "homa_ir": [2.2, float("nan")], "fev1": [3.0, float("nan")], "fvc": [4.0, float("nan")],
        "fev1_fvc": [75.0, float("nan")], "creatinine": [0.8, float("nan")],
        "egfr": [round(egfr_ckd_epi(0.8, 41, "Female")), float("nan")],
        "pack_years": [0.0, 0.0], "smoking_status": ["never", "never"],
        "has_CVD": [0, 0], "has_T2D": [0, 0], "has_CRD": [0, 0], "has_NEURO": [0, 0],
        "fhrs_CVD": [0.5, 0.5], "fhrs_T2D": [0.0, 0.0], "fhrs_CRD": [0.0, 0.0], "fhrs_NEURO": [0.0, 0.0]})
    return patients, relatives, visits

def fails(p, r, v, ranges=SMALL_RANGES):
    return {k: n for k, n in check_hard_rules(p, r, v, ranges).items() if n > 0}

def test_clean_data_has_zero_failures():
    assert fails(*clean()) == {}

def test_rule_1_value_out_of_range():
    p, r, v = clean(); v.loc[1, "sbp"] = 300
    assert fails(p, r, v) == {"1 value inside dictionary range": 1}

def test_rule_1_missing_column_is_a_failure_not_skipped():
    p, r, v = clean()
    assert "1 column missing: hba1c" in fails(p, r, v, {**SMALL_RANGES, "hba1c": (4.0, 15.0)})

def test_rule_2_age_matches_birth_year():
    p, r, v = clean(); v.loc[1, "age"] = 50
    assert fails(p, r, v) == {"2 age matches birth year and visit date": 1}

def test_rule_3_visit_dates_move_forward():
    p, r, v = clean(); v.loc[1, "visit_date"] = pd.Timestamp("2020-01-01"); v.loc[1, "age"] = 40
    assert "3 visit dates move forward" in fails(p, r, v)

def test_rule_4_pack_years_never_go_down():
    p, r, v = clean(); v.loc[0, "pack_years"] = 5.0
    assert "4 pack-years never go down" in fails(p, r, v)

def test_rule_5_diagnosis_stays():
    p, r, v = clean(); v.loc[0, "has_CVD"] = 1
    assert "5 diagnosis stays in later visits" in fails(p, r, v)

def test_rule_6_relative_diagnosis_inside_their_life():
    p, r, v = clean(); r.loc[0, "death_year"] = 2005
    assert "6 relative diagnosis inside their lifetime" in fails(p, r, v)

def test_rule_7_smoking_never_goes_back_to_never():
    p, r, v = clean(); v.loc[0, "smoking_status"] = "former"
    assert "7 smoking never goes back to never" in fails(p, r, v)

def test_rule_8_gestational_diabetes_only_female():
    p, r, v = clean(); p.loc[0, "sex"] = "Male"; p.loc[0, "gestational_diabetes"] = 1
    assert "8 gestational diabetes only for females" in fails(p, r, v)

def test_rule_9_parents_older_and_degree_matches_relation():
    p, r, v = clean(); r.loc[0, "birth_year"] = 1990
    assert "9 parents older than patient" in fails(p, r, v)
    p, r, v = clean(); r.loc[2, "degree"] = 1
    assert "9 degree matches relation" in fails(p, r, v)

def test_rule_10_computed_fields_match_inputs():
    for col, bad in [("bmi", 30.0), ("homa_ir", 9.9), ("fev1_fvc", 50.0), ("egfr", 20), ("fhrs_CVD", 3.0)]:
        p, r, v = clean(); v.loc[0, col] = bad
        assert f"10 computed field matches: {col}" in fails(p, r, v), col

def test_rule_10_fhrs_ignores_a_relatives_undiagnosed_diabetes():
    p, r, v = clean()
    r["diagnosed"] = [True, None, None]
    extra = pd.DataFrame({"patient_id": [0], "relative_no": [2], "relation": ["mother"], "degree": [1], "sex": ["F"],
                          "birth_year": [1952], "death_year": [None], "disease": ["T2D"], "dx_year": [2005],
                          "dx_age": [53], "confirmed": [True], "diagnosed": [False]})
    r = pd.concat([r, extra], ignore_index=True)
    r["death_year"] = r["death_year"].astype(float)          # as read from the CSV (NaN, not None)
    assert fails(p, r, v) == {}          # fhrs_T2D stays 0.0: the patient cannot know undiagnosed diabetes

.

.

.

.

.

.

... (5 outputs left out)

.

.

                                                                                [100%]


13 passed in 0.34s


In [6]:
# RUN  (about 30-60 seconds)
HARD_RULES = check_hard_rules(PATIENTS, RELATIVES, VISITS)
print_report(HARD_RULES)

PASS  1 value inside dictionary range
PASS  2 age matches birth year and visit date
PASS  3 visit dates move forward
PASS  4 pack-years never go down
PASS  5 diagnosis stays in later visits
PASS  6 relative diagnosis inside their lifetime
PASS  7 smoking never goes back to never
PASS  8 gestational diabetes only for females
PASS  9 parents older than patient
PASS  9 grandparents older than parent
PASS  9 degree matches relation
PASS  10 computed field matches: bmi
PASS  10 computed field matches: homa_ir
PASS  10 computed field matches: fev1_fvc
PASS  10 computed field matches: egfr
PASS  10 computed field matches: fhrs_CVD
PASS  10 computed field matches: fhrs_T2D
PASS  10 computed field matches: fhrs_CRD
PASS  10 computed field matches: fhrs_NEURO

ALL HARD RULES PASS


True

## Cell 3: Population check
Compares the synthetic patients with Philippine figures. PASS = the 95% confidence interval of the synthetic share touches the target range. A single-number target gets ±20% around it (author-defined). FAIL here means: change a setting in Notebook 1, run it again, then run this notebook again.

**CVD (physician decision, Nov 5, 2026):** the CVD label now counts hypertension as well as heart disease and stroke. GBD counts heart disease + stroke only, so the cardiovascular *events* are checked against GBD on their own. If the whole label fails against GBD alone, it is compared with GBD heart disease + stroke plus the FNRI hypertension figure (17-23%) instead: the union is at least its larger part and at most the sum of the parts. The data are never changed to make a check pass.

**Cell 3b** reports the physician-approved shares for gestational diabetes, acanthosis and allergic rhinitis (no longer inputs since Phase 0). Since Phase 0b, acanthosis and allergic rhinitis are *calibrated* to these shares in Notebook 1 (like `prevalence_factor` to GBD); gestational diabetes is report only.

**BMI (approved Nov 6):** last visit against DOST-FNRI (2022), *2021 Expanded National Nutrition Survey: National Results Dissemination*. Adults 20-59, WHO Asia-Pacific bands: <18.5 7.2%, 18.5-22.9 34.3%, 23-24.9 18.3%, 25-29.9 30.2%, >=30 10.0%. Adults 60+ (WHO cut-offs only): <18.5 11.8%, 18.5-24.9 56.6%, 25-29.9 25.4%, >=30 6.2%. Notebook 1 calibrates BMI to these (like `prevalence_factor` to GBD); all bins are counted in the PASS/FAIL total.

In [7]:
# CODE
# ---------------------------------------------------------------------------
# Cell 3: population check
# Rule: PASS when the 95% confidence interval of the synthetic share touches
# the target range. A point target gets +/- REL_TOL around it (author-defined),
# because surveys use other age mixes and their own sampling error.
# ---------------------------------------------------------------------------
REL_TOL = 0.20

GBD_CAUSES = {
    "CVD": ["Ischemic heart disease", "Stroke"],
    "T2D": ["Diabetes mellitus type 2"],
    "CRD": ["Asthma", "Chronic obstructive pulmonary disease"],
}

def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes out of n."""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

def check_share(name, flags, lo, hi, source):
    flags = flags.dropna().astype(bool)
    k, n = int(flags.sum()), len(flags)
    ci_lo, ci_hi = wilson(k, n)
    ok = ci_hi >= lo and ci_lo <= hi
    return {"check": name, "synthetic": k / n if n else np.nan, "ci_95": f"{ci_lo:.1%}-{ci_hi:.1%}",
            "target": f"{lo:.1%}-{hi:.1%}", "source": source, "result": "PASS" if ok else "FAIL"}

def last_visit(visits, patients):
    """Each patient's latest visit + sex. Takes only 'sex' from patients, so no _x/_y duplicate columns."""
    last = visits.sort_values(["patient_id", "visit_date"]).groupby("patient_id").tail(1)
    return last.merge(patients[["patient_id", "sex"]], on="patient_id").reset_index(drop=True)

def gbd_expected_share(last, gbd, causes):
    """Share we would expect if each patient had GBD 2023 prevalence for their age group and sex."""
    g = gbd[(gbd["measure_name"] == "Prevalence") & (gbd["metric_name"] == "Rate") & gbd["cause_name"].isin(causes)]
    g = g.assign(age_lo=g["age_name"].str.extract(r"^(\d+)")[0].astype(float))
    prev = (g.groupby(["sex_name", "age_lo"])["val"].sum() / 100_000).clip(upper=1.0)
    keys = list(zip(last["sex"], (last["age"] // 5 * 5).clip(15, 95).astype(float)))
    return float(np.mean([prev.get(k, np.nan) for k in keys]))

def gbd_column(dom, last):
    """GBD counts diagnosed AND undiagnosed diabetes, so T2D is compared with has_T2D_any when it exists."""
    return "has_T2D_any" if dom == "T2D" and "has_T2D_any" in last.columns else f"has_{dom}"

def population_checks(patients, visits, gbd, rel_tol=REL_TOL):
    last = last_visit(visits, patients)
    around = lambda x: (x * (1 - rel_tol), x * (1 + rel_tol))
    rows = []
    for dom, causes in GBD_CAUSES.items():
        exp = gbd_expected_share(last, gbd, causes)
        row = check_share(f"{dom} vs GBD 2023 (age/sex matched)", last[gbd_column(dom, last)], *around(exp), "GBD 2023")
        if dom == "CVD" and "has_HTN" in last:
            # Physician decision (Nov 5, 2026): the CVD label = cardiovascular event OR hypertension.
            # GBD counts heart disease + stroke only, so the events are checked against GBD on their own ...
            rows.append(check_share("CVD events only (IHD + stroke) vs GBD 2023", last["has_CV_event"],
                                    *around(exp), "GBD 2023"))
            # ... and, if the whole label fails against GBD alone (as expected), it is compared with
            # GBD heart disease + stroke plus FNRI hypertension (17-23%): a union is at least its larger
            # part and at most the sum of its parts.
            if row["result"] == "FAIL":
                lo, hi = around(exp)
                row = check_share("CVD label (event or hypertension) vs GBD + FNRI hypertension", last["has_CVD"],
                                  max(lo, 0.17), hi + 0.23, "GBD 2023 + FNRI 2018-19")
        rows.append(row)
    women, men = last[last["sex"] == "Female"], last[last["sex"] == "Male"]
    rows.append(check_share("Diabetes, women", women["has_T2D"], *around(0.078), "NOH 2023-2028"))
    rows.append(check_share("Diabetes, men", men["has_T2D"], *around(0.079), "NOH 2023-2028"))
    rows.append(check_share("Hypertension (SBP>=140 or DBP>=90)", (last["sbp"] >= 140) | (last["dbp"] >= 90),
                            0.17, 0.23, "FNRI 2018-19"))
    rows.append(check_share("Current smokers", last["smoking_status"] == "current", *around(0.204), "FNRI 2018-19"))
    # Guide: PHQ-9 >= 10 catches more than clinical depression, so somewhat higher than 3.3% is fine;
    # "many times higher" is not. Upper limit 3x is author-defined.
    rows.append(check_share("Current depression (PHQ-9>=10)", last["phq9"] >= 10, 0.033, 0.099, "WHO 2021"))
    return pd.concat([pd.DataFrame(rows), bmi_checks(patients, visits, rel_tol)], ignore_index=True)

# Cell 3b: physician-approved shares, kept as CHECKS (Phase 0: no longer inputs, so they cannot leak a
# future diagnosis). Report only: the relative risks are NOT tuned to hit them.
# (flag, disease, approved share in patients who ever get it, approved share in the others, sex)
APPROVED_SHARES = [("gestational_diabetes", "T2D", 0.10, 0.03, "Female"),
                   ("acanthosis_nigricans", "T2D", 0.40, 0.05, None),
                   ("allergic_rhinitis", "asthma", 0.60, 0.15, None)]

def approved_share_report(patients, checks=APPROVED_SHARES):
    rows = []
    for flag, disease, case_share, other_share, sex in checks:
        p = patients if sex is None else patients[patients["sex"] == sex]
        ever = p[f"true_dx_year_{disease}"].notna()
        for group, mask, approved in ((f"{disease} cases", ever, case_share), (f"no {disease}", ~ever, other_share)):
            k, n = int(p.loc[mask, flag].sum()), int(mask.sum())
            lo, hi = wilson(k, n)
            rows.append({"flag": flag, "group": group + (" (women)" if sex else ""), "n": n,
                         "synthetic": k / n if n else np.nan, "ci_95": f"{lo:.1%}-{hi:.1%}", "approved": approved})
    return pd.DataFrame(rows)

# BMI against DOST-FNRI (2022), 2021 Expanded National Nutrition Survey: National Results Dissemination.
# Source approved Nov 6. Adults 20-59, WHO Asia-Pacific bands: <18.5 = chronic energy deficiency 7.2%;
# 25-29.9 = 40.2% (BMI >= 25) - 10.0% (BMI >= 30); 18.5-22.9 = the rest. Adults 60+: WHO cut-offs only.
ENNS_BMI = [((0, 18.5), 0.072), ((18.5, 23.0), 0.343), ((23.0, 25.0), 0.183), ((25.0, 30.0), 0.302), ((30.0, 99.0), 0.100)]
ENNS_BMI_60 = [((0, 18.5), 0.118), ((18.5, 25.0), 0.566), ((25.0, 30.0), 0.254), ((30.0, 99.0), 0.062)]

def bmi_checks(patients, visits, rel_tol=REL_TOL, groups=((20, 59, ENNS_BMI), (60, 120, ENNS_BMI_60))):
    last = last_visit(visits, patients)
    rows = []
    for a, b, targets in groups:
        people = last[last["age"].between(a, b)]
        label = f"adults {a}-{b}" if b < 120 else f"adults {a}+"
        for (lo, hi), share in targets:
            name = f"BMI < {hi}" if lo == 0 else f"BMI >= {lo}" if hi == 99.0 else f"BMI {lo}-{hi - 0.1:.1f}"
            rows.append(check_share(f"{name}, {label}", people["bmi"].between(lo, hi, inclusive="left"),
                                    share * (1 - rel_tol), share * (1 + rel_tol), "FNRI ENNS 2021"))
    return pd.DataFrame(rows)

def print_population_report(table):
    show = table.assign(synthetic=table["synthetic"].map("{:.1%}".format))
    print(show[["check", "synthetic", "ci_95", "target", "source", "result"]].to_string(index=False))
    n_fail = int((table["result"] == "FAIL").sum())
    print("\nALL POPULATION CHECKS PASS" if n_fail == 0 else f"\n{n_fail} CHECK(S) FAILED: change settings in Notebook 1")

In [8]:
# load GBD for the checks and the tests
GBD = pd.read_csv("data/gbd_2023_ph_rates.csv")

In [9]:
%%ipytest -q
# TEST
import pytest

def test_wilson_interval_matches_known_value():
    lo, hi = wilson(50, 100)
    assert lo == pytest.approx(0.404, abs=0.001) and hi == pytest.approx(0.596, abs=0.001)

def test_check_share_passes_when_interval_touches_target():
    row = check_share("x", pd.Series([True] * 20 + [False] * 80), 0.17, 0.23, "src")
    assert row["result"] == "PASS"

def test_check_share_fails_when_far_from_target():
    row = check_share("x", pd.Series([True] * 90 + [False] * 10), 0.17, 0.23, "src")
    assert row["result"] == "FAIL"

def test_last_visit_is_one_row_per_patient_without_duplicate_columns():
    visits = pd.DataFrame({"patient_id": [1, 1, 2], "visit_date": pd.to_datetime(["2021-01-01", "2023-01-01", "2022-01-01"]),
                           "sbp": [110, 150, 120], "smoking_status": ["never"] * 3})
    patients = pd.DataFrame({"patient_id": [1, 2], "sex": ["Male", "Female"], "smoking_status": ["never"] * 2})
    last = last_visit(visits, patients)
    assert list(last["sbp"]) == [150, 120]
    assert "smoking_status" in last and not any(c.endswith(("_x", "_y")) for c in last)

def test_gbd_expected_share_matches_age_and_sex_and_adds_causes():
    gbd = pd.DataFrame({"measure_name": ["Prevalence"] * 3, "metric_name": ["Rate"] * 3,
                        "sex_name": ["Male", "Male", "Female"], "age_name": ["40-44 years"] * 3,
                        "cause_name": ["Asthma", "Chronic obstructive pulmonary disease", "Asthma"],
                        "val": [10_000, 5_000, 99_000]})
    last = pd.DataFrame({"age": [42, 43], "sex": ["Male", "Male"]})
    assert gbd_expected_share(last, gbd, ["Asthma", "Chronic obstructive pulmonary disease"]) == pytest.approx(0.15)

def test_population_checks_has_every_check_and_a_result():
    table = population_checks(PATIENTS, VISITS, GBD)
    n = (9 if "has_HTN" in VISITS else 8) + len(ENNS_BMI) + len(ENNS_BMI_60)   # + CVD events (Nov 5) + BMI (Nov 6)
    assert len(table) == n and set(table["result"]) <= {"PASS", "FAIL"}

def test_bmi_check_uses_adults_20_to_59_and_both_cutoffs():
    visits = pd.DataFrame({"patient_id": [1, 2, 3], "visit_date": pd.to_datetime(["2022-01-01"] * 3),
                           "age": [30, 45, 70], "bmi": [24.0, 26.0, 30.0]})
    patients = pd.DataFrame({"patient_id": [1, 2, 3], "sex": ["Male", "Female", "Male"]})
    t = bmi_checks(patients, visits).set_index("check")
    assert t.loc["BMI 23.0-24.9, adults 20-59", "synthetic"] == 0.5       # the 70-year-old is left out
    assert t.loc["BMI 25.0-29.9, adults 20-59", "synthetic"] == 0.5
    assert t.loc["BMI < 18.5, adults 20-59", "synthetic"] == 0.0
    assert sum(s for _, s in ENNS_BMI) == pytest.approx(1.0) and sum(s for _, s in ENNS_BMI_60) == pytest.approx(1.0)
    assert bmi_checks(patients, visits).set_index("check").loc["BMI >= 30.0, adults 60+", "synthetic"] == 1.0

def test_approved_shares_are_reported_for_cases_and_others():
    patients = pd.DataFrame({"sex": ["Female", "Female", "Male", "Male"], "true_dx_year_T2D": [2022, None, 2023, None],
                             "true_dx_year_asthma": [None, 2024, None, None], "gestational_diabetes": [1, 0, 0, 0],
                             "acanthosis_nigricans": [1, 0, 1, 1], "allergic_rhinitis": [0, 1, 0, 0]})
    t = approved_share_report(patients).set_index(["flag", "group"])
    assert t.loc[("gestational_diabetes", "T2D cases (women)"), "synthetic"] == 1.0      # women only
    assert t.loc[("acanthosis_nigricans", "no T2D"), "synthetic"] == 0.5
    assert t.loc[("allergic_rhinitis", "asthma cases"), "approved"] == 0.60

def test_gbd_t2d_check_reads_all_diabetes_when_the_column_exists():
    assert gbd_column("T2D", pd.DataFrame(columns=["has_T2D", "has_T2D_any"])) == "has_T2D_any"
    assert gbd_column("T2D", pd.DataFrame(columns=["has_T2D"])) == "has_T2D"
    assert gbd_column("CVD", pd.DataFrame(columns=["has_CVD", "has_T2D_any"])) == "has_CVD"

.

.

.

.

.

.

.

.

.

                                                                                    [100%]


9 passed in 0.46s


In [10]:
# RUN
POPULATION = population_checks(PATIENTS, VISITS, GBD)
print_population_report(POPULATION)

print("\nCell 3b: physician-approved shares (acanthosis and rhinitis calibrated to them in Notebook 1; GDM report only)")
APPROVED = approved_share_report(PATIENTS)
print(APPROVED.assign(synthetic=APPROVED["synthetic"].map("{:.1%}".format),
                      approved=APPROVED["approved"].map("{:.0%}".format)).to_string(index=False))

                                                       check synthetic       ci_95      target                  source result
                  CVD events only (IHD + stroke) vs GBD 2023     10.8% 10.4%-11.2%  8.6%-12.9%                GBD 2023   PASS
CVD label (event or hypertension) vs GBD + FNRI hypertension     20.2% 19.7%-20.8% 17.0%-35.9% GBD 2023 + FNRI 2018-19   PASS
                           T2D vs GBD 2023 (age/sex matched)     15.6% 15.1%-16.1% 13.1%-19.7%                GBD 2023   PASS
                           CRD vs GBD 2023 (age/sex matched)     12.9% 12.4%-13.3% 12.0%-18.0%                GBD 2023   PASS
                                             Diabetes, women      7.1%   6.6%-7.6%   6.2%-9.4%           NOH 2023-2028   PASS
                                               Diabetes, men      8.0%   7.5%-8.6%   6.3%-9.5%           NOH 2023-2028   PASS
                          Hypertension (SBP>=140 or DBP>=90)     20.3% 19.7%-20.8% 17.0%-23.0%            FNRI 2018-19

## Cell 4: Parameter recovery (Leu et al., 2007)
Notebook 1 doubled a person's yearly risk once a parent or sibling had the same disease (`family_multiplier = 2.0`). This cell measures that number back from the data without using it. PASS = the pooled rate ratio is between 1.7 and 2.3.

In [11]:
# CODE
# ---------------------------------------------------------------------------
# Cell 4: parameter recovery (Leu et al., 2007)
# Notebook 1 made a person's yearly risk x2.0 once a parent or sibling had the
# same disease. Here we measure it back from the data, without using that 2.0.
# Measure: incidence rate ratio (cases per person-year with vs without an
# affected first-degree relative), Mantel-Haenszel, by disease, sex and
# 5-year age group, so older people (who more often have ill relatives AND
# higher base risk) do not fake a bigger effect.
# ---------------------------------------------------------------------------
DISEASES = ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]
RECOVERY_RANGE = (1.7, 2.3)          # same range as the Coding Guide test

def simulate_family_diseases(relatives, cfg, rng, rate_fn, diseases, end_year=2026):
    """Copied from Notebook 1, Part B (used by the tests)."""
    first = min(r["birth_year"] for r in relatives)
    for year in range(first, end_year + 1):
        for r in relatives:
            age = year - r["birth_year"]
            if age < 0 or (r.get("death_year") and year > r["death_year"]):
                continue
            for d in diseases:
                if any(c["disease"] == d for c in r["conditions"]):
                    continue
                rate = rate_fn(d, age, r["sex"]) * r.get("risk_mult", {}).get(d, 1.0)
                if any(o is not r and o["degree"] == 1 and
                       any(c["disease"] == d and c["year"] < year for c in o["conditions"])
                       for o in relatives):
                    rate *= cfg["family_multiplier"]
                if rng.random() < min(rate, 0.99):
                    r["conditions"].append({"disease": d, "year": year, "age": age, "confirmed": True})

def person_years(patients, relatives, diseases=DISEASES, end_year=2027, min_age=15):
    """One row per patient, disease and year at risk: stratum, exposed (0/1), case (0/1)."""
    first = relatives[relatives["degree"] == 1].dropna(subset=["disease"])
    earliest = first.groupby(["patient_id", "disease"])["dx_year"].min().to_dict()
    rows = []
    for p in patients.itertuples(index=False):
        for d in diseases:
            dx = getattr(p, f"true_dx_year_{d}", None)
            dx = None if dx is None or pd.isna(dx) else int(dx)
            rel = earliest.get((p.patient_id, d))
            last = end_year if dx is None else min(dx, end_year)
            for year in range(int(p.birth_year) + min_age, last + 1):
                age = year - int(p.birth_year)
                rows.append((f"{d}|{p.sex}|{age - age % 5}",
                             int(rel is not None and rel < year), int(dx == year)))
    return pd.DataFrame(rows, columns=["stratum", "exposed", "case"])

def mh_irr(py):
    """Mantel-Haenszel incidence rate ratio over strata."""
    g = py.groupby(["stratum", "exposed"])["case"].agg(["sum", "count"]).unstack(fill_value=0)
    a, b = g[("sum", 1)], g[("sum", 0)]
    t1, t0 = g[("count", 1)], g[("count", 0)]
    t = t1 + t0
    den = (b * t1 / t).sum()
    return float((a * t0 / t).sum() / den) if den > 0 else np.nan

def recovery_report(patients, relatives, rng_range=RECOVERY_RANGE):
    py = person_years(patients, relatives)
    py["disease"] = py["stratum"].str.split("|").str[0]
    rows = [{"disease": d, "cases": int(g["case"].sum()), "irr": mh_irr(g)} for d, g in py.groupby("disease")]
    rows.append({"disease": "ALL (pooled)", "cases": int(py["case"].sum()), "irr": mh_irr(py)})
    table = pd.DataFrame(rows)
    pooled = table.iloc[-1]["irr"]
    ok = rng_range[0] < pooled < rng_range[1]
    print(table.to_string(index=False, formatters={"irr": "{:.2f}".format}))
    print(f"\nFamily multiplier put in: 2.0   measured back: {pooled:.2f}   "
          f"target {rng_range[0]}-{rng_range[1]}   {'PASS' if ok else 'FAIL'}")
    print("(Per-disease numbers move more because each has fewer cases; the pooled number is the check.)")
    return table, ok

In [12]:
%%ipytest -q
# TEST
import pytest

# ---------- Cell 4 ----------
def test_mh_irr_on_a_hand_made_table():
    # one stratum: exposed 4 cases in 100 years, unexposed 2 cases in 100 years -> IRR 2.0
    py = pd.DataFrame({"stratum": ["s"] * 200, "exposed": [1] * 100 + [0] * 100,
                       "case": [1] * 4 + [0] * 96 + [1] * 2 + [0] * 98})
    assert mh_irr(py) == pytest.approx(2.0)

def test_mh_irr_removes_age_confounding():
    # same IRR 2 in each age stratum, but exposed people are mostly old (higher base rate)
    young = pd.DataFrame({"stratum": "young", "exposed": [1] * 100 + [0] * 900, "case": [1] * 2 + [0] * 98 + [1] * 9 + [0] * 891})
    old = pd.DataFrame({"stratum": "old", "exposed": [1] * 900 + [0] * 100, "case": [1] * 90 + [0] * 810 + [1] * 5 + [0] * 95})
    py = pd.concat([young, old])
    crude = py[py.exposed == 1].case.mean() / py[py.exposed == 0].case.mean()
    assert crude > 3 and mh_irr(py) == pytest.approx(2.0, abs=0.01)

def _flat_families(multiplier, n=3000, seed=7):
    rng = np.random.default_rng(seed)
    flat = lambda d, age, sex: 0.01 if age >= 30 else 0.0
    pats, rels = [], []
    for pid in range(n):
        fam = [{"degree": 1, "sex": "F", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "M", "birth_year": 1950, "conditions": []},
               {"degree": 0, "sex": "F", "birth_year": 1975, "conditions": []}]
        simulate_family_diseases(fam, {"family_multiplier": multiplier}, rng, flat, ["T2D"], end_year=2027)
        me = {c["disease"]: c["year"] for c in fam[2]["conditions"]}
        pats.append({"patient_id": pid, "sex": "Female", "birth_year": 1975, "true_dx_year_T2D": me.get("T2D")})
        for r in fam[:2]:
            for c in r["conditions"]:
                rels.append({"patient_id": pid, "degree": 1, "disease": c["disease"], "dx_year": c["year"]})
    return pd.DataFrame(pats), pd.DataFrame(rels, columns=["patient_id", "degree", "disease", "dx_year"])

def test_recovers_multiplier_2_from_simulated_families():
    p, r = _flat_families(2.0)
    assert 1.7 < mh_irr(person_years(p, r, ["T2D"])) < 2.3

def test_recovers_no_effect_when_multiplier_is_1():
    p, r = _flat_families(1.0)
    assert 0.8 < mh_irr(person_years(p, r, ["T2D"])) < 1.25

.

.

.

.

                                                                                         [100%]


4 passed in 1.42s


In [13]:
# RUN  (about 5-10 seconds)
RECOVERY, RECOVERY_OK = recovery_report(PATIENTS, RELATIVES)

     disease  cases  irr
        COPD   1427 1.98
         CVD   2714 1.94
         GAD    136 2.22
         T2D   3469 1.98
      asthma   1728 1.76
  depression   1439 2.00
ALL (pooled)  10913 1.94

Family multiplier put in: 2.0   measured back: 1.94   target 1.7-2.3   PASS
(Per-disease numbers move more because each has fewer cases; the pooled number is the check.)


## Cell 5: Doctor review sample
20 patients (10 with a disease, 10 without), same 20 every run. Saved to `output/doctor_review_v2.csv` with two empty columns for the doctor.

`output/doctor_review.csv` holds the 20 patients the physician already rated (Nov 5, 2026). It is **never overwritten**: `save_review` refuses that file name.

In [14]:
# CODE
# ---------------------------------------------------------------------------
# Cell 5: doctor review sample
# 20 patients, half with a disease and half without (so the doctor sees both),
# fixed seed so the same 20 come out every run. One row per visit, with the
# family history written as text and two empty columns for the doctor.
# ---------------------------------------------------------------------------
REVIEW_COLUMNS = ["sbp", "dbp", "heart_rate", "bmi", "waist_cm", "hba1c", "fpg", "ldl", "hdl",
                  "fev1_fvc", "spo2", "mmrc", "phq9", "gad7", "sleep_hours", "smoking_status", "pack_years",
                  "fhrs_band_CVD", "fhrs_band_T2D", "fhrs_band_CRD", "fhrs_band_NEURO",
                  "has_CVD", "has_T2D", "has_CRD", "has_NEURO"]
REVIEW_COLUMNS_NEW = ["has_CV_event", "has_HTN"]     # since the Nov 5 CVD label, when present
PHYSICIAN_RATED = "doctor_review.csv"                # rated by the physician on Nov 5, 2026: never overwrite

def save_review(review, path):
    from pathlib import Path
    path = Path(path)
    if path.name == PHYSICIAN_RATED:
        raise ValueError(f"{path} holds the physician's ratings and must never be overwritten")
    review.to_csv(path, index=False)
    return path

def family_text(rel_rows):
    parts = []
    for r in rel_rows.itertuples(index=False):
        if pd.isna(r.disease):
            continue
        age = "age unknown" if pd.isna(r.dx_age) else f"at {int(r.dx_age)}"
        sure = "confirmed" if str(r.confirmed) == "True" else "not sure"
        died = "" if pd.isna(r.death_year) else f", died {int(r.death_year)}"
        hidden = ", undiagnosed: patient does not know" if str(getattr(r, "diagnosed", True)) == "False" else ""
        parts.append(f"{r.relation}: {r.disease} {age} ({sure}{died}{hidden})")
    return "; ".join(parts) if parts else "no known family disease"

def doctor_review_sample(patients, relatives, visits, training, n=20, seed=1):
    dx_cols = [c for c in patients if c.startswith("true_dx_year_") and c != "true_dx_year_HTN"]
    sick = patients[dx_cols].notna().any(axis=1)
    rng = np.random.default_rng(seed)
    pick = sorted(list(rng.choice(patients.loc[sick, "patient_id"], n // 2, replace=False)) +
                  list(rng.choice(patients.loc[~sick, "patient_id"], n - n // 2, replace=False)))
    p = patients.set_index("patient_id")
    fam = {pid: family_text(g) for pid, g in relatives[relatives["patient_id"].isin(pick)].groupby("patient_id")}
    lab = (training[training["patient_id"].isin(pick)]
           .assign(t=lambda d: d["domain"] + "=" + d["label"].astype(str))
           .groupby("patient_id")["t"].agg(", ".join).to_dict())
    own = {pid: "; ".join(f"{c.replace('true_dx_year_', '')} {int(y)}" for c, y in p.loc[pid, dx_cols].items() if pd.notna(y))
           or "none" for pid in pick}
    v = visits[visits["patient_id"].isin(pick)].sort_values(["patient_id", "visit_date"])
    out = v[["patient_id", "visit_no", "visit_date", "age"] + REVIEW_COLUMNS
            + [c for c in REVIEW_COLUMNS_NEW if c in v]].copy()
    out.insert(1, "sex", out["patient_id"].map(p["sex"]))
    out["patient_diagnoses (year)"] = out["patient_id"].map(own)
    out["family_history"] = out["patient_id"].map(fam).fillna("no known family disease")
    out["training_labels"] = out["patient_id"].map(lab).fillna("")
    out["doctor_realistic_yes_no"] = ""
    out["doctor_comment"] = ""
    return out.reset_index(drop=True)

In [15]:
%%ipytest -q
# TEST
import pytest

# ---------- Cell 5 ----------
def test_review_sample_has_20_patients_half_with_a_disease():
    s = doctor_review_sample(PATIENTS, RELATIVES, VISITS, TRAINING, n=20, seed=1)
    ids = s["patient_id"].unique()
    assert len(ids) == 20
    sick = PATIENTS.set_index("patient_id").loc[ids, [c for c in PATIENTS if c.startswith("true_dx_year_") and c != "true_dx_year_HTN"]].notna().any(axis=1)
    assert sick.sum() == 10

def test_review_sample_is_the_same_every_run():
    a = doctor_review_sample(PATIENTS, RELATIVES, VISITS, TRAINING, n=20, seed=1)
    b = doctor_review_sample(PATIENTS, RELATIVES, VISITS, TRAINING, n=20, seed=1)
    assert a.equals(b)

def test_review_sample_has_family_text_and_blank_doctor_columns():
    s = doctor_review_sample(PATIENTS, RELATIVES, VISITS, TRAINING, n=20, seed=1)
    assert {"family_history", "doctor_realistic_yes_no", "doctor_comment"} <= set(s.columns)
    assert s["doctor_comment"].eq("").all()
    assert s["family_history"].str.len().gt(0).all()

def test_family_text_marks_undiagnosed_diabetes():
    rows = pd.DataFrame({"relation": ["mother"], "disease": ["T2D"], "dx_age": [50], "confirmed": [True],
                         "death_year": [None], "diagnosed": [False]})
    assert "undiagnosed" in family_text(rows)

def test_the_physician_rated_file_is_never_overwritten(tmp_path):
    import pytest
    (tmp_path / "doctor_review.csv").write_text("rated")
    with pytest.raises(ValueError):
        save_review(pd.DataFrame({"a": [1]}), tmp_path / "doctor_review.csv")
    assert (tmp_path / "doctor_review.csv").read_text() == "rated"
    assert save_review(pd.DataFrame({"a": [1]}), tmp_path / "doctor_review_v2.csv").exists()

.

.

.

.

.

                                                                                        [100%]


5 passed in 0.11s


In [16]:
# RUN  (doctor review sample: only from output/, never from the second seed)
if FOLDER == "output":
    REVIEW = doctor_review_sample(PATIENTS, RELATIVES, VISITS, TRAINING, n=20, seed=1)
    saved = save_review(REVIEW, "output/doctor_review_v2.csv")
    print(f"Saved {saved}: {REVIEW['patient_id'].nunique()} patients, {len(REVIEW)} visit rows")
    print("output/doctor_review.csv (rated by the physician on Nov 5) is left as it was.")
    print("Give the new file to the doctor. They fill in 'doctor_realistic_yes_no' and 'doctor_comment'.")
else:
    print(f"{FOLDER}: no doctor review sample (only from output/)")


Saved output/doctor_review_v2.csv: 20 patients, 140 visit rows
output/doctor_review.csv (rated by the physician on Nov 5) is left as it was.
Give the new file to the doctor. They fill in 'doctor_realistic_yes_no' and 'doctor_comment'.
